# CTI NER — HuggingFace Hub model demo

Loads the two published checkpoints and runs them on real sentences pulled from CISA advisories in `data/alerts/`:

- [`pzryathzsdhc/cti-ner-roberta-large`](https://huggingface.co/pzryathzsdhc/cti-ner-roberta-large)
- [`pzryathzsdhc/cti-ner-securebert`](https://huggingface.co/pzryathzsdhc/cti-ner-securebert)

**Checkpoint provenance:** `cti-ner-securebert` was re-pushed 2026-07-24 after retraining on the clean LLM gold (Claude Sonnet 5-proposed, human-curated eval; strict micro F1 0.71 on the held-out test split). `cti-ner-roberta-large` still holds the *old* dictionary/GLiNER-ensemble silver-label weights pending its own re-push — treat its predictions as a demo of the prior pipeline, not the paper's numbers (see `CLAUDE.md`, `paper/capstone/capstone.tex`).

Inference replicates the sliding-window word-tagging procedure from `ner/finetune_roberta.py` (`make_windows`, `ChunkWindows`, `predict_chunk_spans`, `bio_to_spans`) so results match what the eval harness would produce, rather than a naive single-pass tokenizer call.

In [5]:
%pip install -q transformers torch pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
import re

import pandas as pd
import torch
from IPython.display import HTML, display
from transformers import AutoModelForTokenClassification, AutoTokenizer

MODEL_IDS = {
    "RoBERTa-large": "pzryathzsdhc/cti-ner-roberta-large",
    "SecureBERT": "pzryathzsdhc/cti-ner-securebert",
}

# The 16 CTI labels this project's pipeline annotates for (ner/frontier_llm_utils.py:LABELS).
# Each published checkpoint's own config.label2id is the source of truth at inference time —
# this list is here for reference only.
LABELS = [
    "affected software product", "attack technique id", "attack technique name",
    "cve identifier", "cwe identifier", "email address", "file hash", "file name",
    "file path", "hacking tool", "ip address", "malware family", "tactic",
    "threat actor group", "tlp classification", "yara rule name",
]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device: {DEVICE}")

device: mps


## Windowed word-tagging inference

Ported from `ner/finetune_roberta.py` so predictions match the paper's eval harness: text is split into words, each word wordpiece-tokenized separately, packed into overlapping windows under a 510-wordpiece budget, and each word's final tag is taken from the window where it sits most centrally (`predict_chunk_spans`).

In [7]:
MAX_WORDPIECES = 510  # 512 minus <s> </s>
WINDOW_OVERLAP_WORDS = 50


def bio_to_spans(word_tags):
    """Decode BIO tags (with standard I-after-O repair) to inclusive word spans."""
    spans = []
    start, label = None, None
    for i, tag in enumerate(word_tags):
        if tag.startswith("B-") or (
            tag.startswith("I-") and (label is None or tag[2:] != label)
        ):
            if label is not None:
                spans.append((start, i - 1, label))
            start, label = i, tag[2:]
        elif tag == "O" and label is not None:
            spans.append((start, i - 1, label))
            start, label = None, None
    if label is not None:
        spans.append((start, len(word_tags) - 1, label))
    return spans


def make_windows(word_piece_counts):
    """[start_word, end_word) windows fitting MAX_WORDPIECES, overlapping."""
    n = len(word_piece_counts)
    windows = []
    start = 0
    while start < n:
        budget, end = 0, start
        while end < n and budget + word_piece_counts[end] <= MAX_WORDPIECES:
            budget += word_piece_counts[end]
            end += 1
        if end == start:  # single pathological word longer than the budget
            end = start + 1
        windows.append((start, end))
        if end >= n:
            break
        overlap = min(WINDOW_OVERLAP_WORDS, (end - start) // 4)
        start = max(start + 1, end - overlap)
    return windows


class ChunkWindows:
    """Tokenized sliding windows for one span of raw text (no gold labels)."""

    def __init__(self, words, tokenizer):
        self.words = words
        self.n_words = len(words)
        self.word_pieces = [
            tokenizer(w, add_special_tokens=False)["input_ids"]
            or [tokenizer.unk_token_id]
            for w in words
        ]
        self.windows = make_windows([len(p) for p in self.word_pieces])
        self.cls = tokenizer.cls_token_id
        self.sep = tokenizer.sep_token_id

    def encode(self, w_start, w_end):
        input_ids, word_index = [self.cls], [-1]
        for wi in range(w_start, w_end):
            pieces = self.word_pieces[wi][:MAX_WORDPIECES]
            input_ids.extend(pieces)
            word_index.extend([wi] + [-1] * (len(pieces) - 1))
        input_ids.append(self.sep)
        word_index.append(-1)
        return {"input_ids": input_ids, "word_index": word_index}


@torch.no_grad()
def predict_chunk_spans(model, chunk, id_to_tag, device):
    """Per-word tags from the window where the word is most interior."""
    best_margin = [-1] * chunk.n_words
    word_tag = ["O"] * chunk.n_words
    for ws, we in chunk.windows:
        enc = chunk.encode(ws, we)
        ids = torch.tensor([enc["input_ids"]], device=device)
        mask = torch.ones_like(ids)
        logits = model(input_ids=ids, attention_mask=mask).logits[0]
        pred = logits.argmax(-1).tolist()
        for pos, wi in enumerate(enc["word_index"]):
            if wi < 0:
                continue
            margin = min(wi - ws, we - 1 - wi)
            if margin > best_margin[wi]:
                best_margin[wi] = margin
                word_tag[wi] = id_to_tag[pred[pos]]
    return bio_to_spans(word_tag)


def predict_text(text, model, tokenizer, id_to_tag, device):
    """Whitespace-tokenize (matches ner/annotation_text.py:tokenize), run the
    windowed word-tagger, and map word spans back to character offsets."""
    word_matches = list(re.finditer(r"\S+", text))
    words = [m.group() for m in word_matches]
    chunk = ChunkWindows(words, tokenizer)
    word_spans = predict_chunk_spans(model, chunk, id_to_tag, device)
    entities = []
    for w_start, w_end, label in word_spans:
        start_char = word_matches[w_start].start()
        end_char = word_matches[w_end].end()
        entities.append({
            "label": label,
            "text": text[start_char:end_char],
            "start": start_char,
            "end": end_char,
        })
    return sorted(entities, key=lambda e: e["start"])

## Load both checkpoints

Label space (`id2label`) comes from each checkpoint's own `config.json` — no separate label file needed.

In [8]:
loaded = {}
for name, repo_id in MODEL_IDS.items():
    print(f"loading {name} ({repo_id}) ...")
    tokenizer = AutoTokenizer.from_pretrained(repo_id, add_prefix_space=True)
    model = AutoModelForTokenClassification.from_pretrained(repo_id).to(DEVICE).eval()
    id_to_tag = {int(i): t for i, t in model.config.id2label.items()}
    loaded[name] = {"tokenizer": tokenizer, "model": model, "id_to_tag": id_to_tag}
    n_labels = (len(id_to_tag) - 1) // 2  # exclude "O", B-/I- pairs per label
    print(f"  {n_labels} entity labels, {len(id_to_tag)} BIO tags")

loading RoBERTa-large (pzryathzsdhc/cti-ner-roberta-large) ...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11940.01it/s]


  16 entity labels, 33 BIO tags
loading SecureBERT (pzryathzsdhc/cti-ner-securebert) ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10311.40it/s]


  16 entity labels, 33 BIO tags


## Sample sentences

Real sentences from CISA advisories in `data/alerts/`, chosen to cover a spread of the 16 labels (CVE, ATT&CK technique IDs, malware family, threat actor group, hacking tools).

In [9]:
SAMPLES = [
    {
        "text": (
            "As part of their initial exploit chain [T1190], the threat actors "
            "uploaded a TGZ file [T1105] containing a generic webshell [T1505.003], "
            "discovery script [TA0007], and setuid binary [T1548.001] on the ADC "
            "appliance and conducted SMB scanning on the subnet [T1046]."
        ),
        "source": "data/alerts/aa23-201a/...clean.md",
    },
    {
        "text": (
            "Log4j \u201cLog4Shell\u201d Apache Logging Services vulnerability "
            "(CVE-2021-44228) [T1190],"
        ),
        "source": "data/alerts/aa22-152a/aa22-152a-karakurt-data-extortion-group.clean.md",
    },
    {
        "text": (
            "Upon developing or obtaining access to a compromised system, Karakurt "
            "actors deploy Cobalt Strike beacons to enumerate a network [T1083], "
            "install Mimikatz to pull plain-text credentials [T1078], use ..."
        ),
        "source": "data/alerts/aa22-152a/aa22-152a-karakurt-data-extortion-group.clean.md",
    },
    {
        "text": (
            "Truebot is a botnet that has been used by malicious cyber groups "
            "like CL0P Ransomware Gang to collect and exfiltrate information "
            "from its target victims."
        ),
        "source": "data/alerts/aa23-187a/...clean.md",
    },
    {
        "text": (
            "newer versions allow cyber threat actors to also gain initial access "
            "through exploiting CVE-2022-31199\u2014(a remote code execution "
            "vulnerability in the Netwrix Auditor application), enabling deployment "
            "of the malware at scale within the compromised environment."
        ),
        "source": "data/alerts/aa23-187a/...clean.md",
    },
]

for i, s in enumerate(SAMPLES):
    print(f"[{i}] ({s['source']})\n    {s['text']}\n")

[0] (data/alerts/aa23-201a/...clean.md)
    As part of their initial exploit chain [T1190], the threat actors uploaded a TGZ file [T1105] containing a generic webshell [T1505.003], discovery script [TA0007], and setuid binary [T1548.001] on the ADC appliance and conducted SMB scanning on the subnet [T1046].

[1] (data/alerts/aa22-152a/aa22-152a-karakurt-data-extortion-group.clean.md)
    Log4j “Log4Shell” Apache Logging Services vulnerability (CVE-2021-44228) [T1190],

[2] (data/alerts/aa22-152a/aa22-152a-karakurt-data-extortion-group.clean.md)
    Upon developing or obtaining access to a compromised system, Karakurt actors deploy Cobalt Strike beacons to enumerate a network [T1083], install Mimikatz to pull plain-text credentials [T1078], use ...

[3] (data/alerts/aa23-187a/...clean.md)
    Truebot is a botnet that has been used by malicious cyber groups like CL0P Ransomware Gang to collect and exfiltrate information from its target victims.

[4] (data/alerts/aa23-187a/...clean.md)
  

## Run inference

In [10]:
rows = []
for i, sample in enumerate(SAMPLES):
    for model_name, m in loaded.items():
        entities = predict_text(
            sample["text"], m["model"], m["tokenizer"], m["id_to_tag"], DEVICE
        )
        for e in entities:
            rows.append({"sentence": i, "model": model_name, **e})

results_df = pd.DataFrame(rows)
results_df

,sentence,model,label,text,start,end
0,0,RoBERTa-large,attack technique id,"[T1190],",39,47
1,0,RoBERTa-large,attack technique id,[T1105],86,93
2,0,RoBERTa-large,attack technique id,"[T1505.003],",124,136
3,0,RoBERTa-large,tactic,"[TA0007],",154,163
4,0,RoBERTa-large,attack technique id,[T1548.001],182,193
5,0,RoBERTa-large,attack technique id,[T1046].,256,264
6,0,SecureBERT,attack technique id,"[T1190],",39,47
7,0,SecureBERT,attack technique id,[T1105],86,93
8,0,SecureBERT,attack technique id,"[T1505.003],",124,136
9,0,SecureBERT,tactic,discovery,137,146


## Highlighted view

Same predictions, rendered inline over each sentence per model.

In [11]:
PALETTE = [
    "#ffd54f", "#90caf9", "#a5d6a7", "#ef9a9a", "#ce93d8", "#ffab91",
    "#80cbc4", "#c5e1a5", "#f48fb1", "#b0bec5", "#fff59d", "#9fa8da",
    "#80deea", "#e6ee9c", "#bcaaa4", "#f8bbd0",
]
LABEL_COLOR = {label: PALETTE[i % len(PALETTE)] for i, label in enumerate(LABELS)}


def highlight_html(text, entities):
    html, cursor = [], 0
    for e in entities:
        html.append(text[cursor:e["start"]])
        color = LABEL_COLOR.get(e["label"], "#eeeeee")
        html.append(
            f'<span style="background:{color};border-radius:3px;padding:0 2px;">'
            f'{text[e["start"]:e["end"]]}'
            f'<sub style="font-size:0.7em;">&nbsp;{e["label"]}</sub></span>'
        )
        cursor = e["end"]
    html.append(text[cursor:])
    return "".join(html)


blocks = []
for i, sample in enumerate(SAMPLES):
    blocks.append(f"<h4>[{i}] {sample['source']}</h4>")
    for model_name in loaded:
        ents = results_df[
            (results_df["sentence"] == i) & (results_df["model"] == model_name)
        ].to_dict("records")
        blocks.append(f"<p><b>{model_name}</b><br>{highlight_html(sample['text'], ents)}</p>")

display(HTML("\n".join(blocks)))

## Notes

- All three models are published to HuggingFace Hub: `cti-ner-securebert` (clean LLM-gold retrain, 2026-07-24, strict micro F1 0.71), `cti-ner-gliner` (clean-gold GLiNER v2, 0.560), and `cti-ner-roberta-large` (still the *prior* dictionary-silver weights, re-push pending).
- Same repo IDs are reused across retrains — check each repo's commit history / model card for provenance before comparing numbers.
- Predictions here are qualitative spot-checks, not a scored evaluation — for quantitative numbers see `paper/capstone/capstone.tex` §Results / `data/gold_v2/results/trained-*.json` (paper numbers come from those committed results, not from re-running this notebook).